# DNABERT2 Shared Split Promoter Benchmark

This notebook runs DNABERT2 on the same GSE144621 train/validation/test CSV split used by the CNN benchmark.

Scientific rule: train on `train`, select the threshold on `validation` using MCC, and report final performance on held-out `test`. Do not tune on the test split.

## What Is Different From The CNN Notebooks?

- CNN uses one-hot DNA sequence tensors.
- DNABERT2 uses a pretrained genomic language model tokenizer/encoder.
- The frozen DNABERT2 run caches embeddings and trains only a classifier head.
- The optional fine-tuning run updates DNABERT2 weights and is more compute-heavy.

The dataset split and metrics stay the same so model comparisons remain fair.

In [ ]:
# Colab setup
# Use Runtime > Change runtime type > GPU before running DNABERT2.

from pathlib import Path
import os
import sys

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
BRANCH = "model-dnabert2-benchmark-notebooks"
REPO_DIR = Path("/content/SeqTrainer") if IN_COLAB else Path.cwd()

if IN_COLAB and not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
!git rev-parse --abbrev-ref HEAD
!git rev-parse HEAD

In [ ]:
# Install package dependencies.
# `accelerate` and `einops` are included because DNABERT2 remote code commonly needs them in Colab.

if IN_COLAB:
    !python -m pip install -q --upgrade pip setuptools wheel
    !python -m pip install -q -e ".[torch]" accelerate einops

import numpy as np
import pandas as pd
import torch

print("python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
# Prepare the shared promoter CSV split.
# The notebook first tries Google Drive if mounted, then falls back to the bundled repo ZIP.

from pathlib import Path
import shutil
import zipfile

DATA_DIR = REPO_DIR / "data" / "promoter_classification"
DATA_DIR.mkdir(parents=True, exist_ok=True)

split_file_names = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}

def files_ready():
    return all((DATA_DIR / name).exists() for name in split_file_names.values())

def try_mount_drive():
    if not IN_COLAB:
        return None
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        return Path("/content/drive/MyDrive")
    except Exception as exc:
        print("Drive mount skipped or failed:", exc)
        return None

def copy_from_drive(my_drive):
    if my_drive is None:
        return False
    candidates = [
        my_drive / "AIxBio" / "Promoter Classification" / "Data",
        my_drive / "AI BIO" / "Promoter Classification" / "Data",
        my_drive / "AI*BIO" / "Promoter Classification" / "Data",
    ]
    for drive_dir in candidates:
        if all((drive_dir / name).exists() for name in split_file_names.values()):
            for name in split_file_names.values():
                shutil.copy2(drive_dir / name, DATA_DIR / name)
            print("Copied split files from Drive:", drive_dir)
            return True
    return False

def extract_from_repo_zip():
    zip_path = REPO_DIR / "data" / "data_DNABERT" / "promoter_classification_DNABERT.zip"
    if not zip_path.exists():
        return False
    with zipfile.ZipFile(zip_path) as zf:
        names = set(zf.namelist())
        for name in split_file_names.values():
            if name not in names:
                raise FileNotFoundError(f"{name} is missing from {zip_path}")
            with zf.open(name) as src, (DATA_DIR / name).open("wb") as dst:
                shutil.copyfileobj(src, dst)
    print("Extracted split files from repo ZIP:", zip_path)
    return True

if not files_ready():
    mounted_drive = try_mount_drive()
    if not copy_from_drive(mounted_drive):
        extract_from_repo_zip()

if not files_ready():
    raise FileNotFoundError(f"Could not prepare all split files in {DATA_DIR}")

for split, name in split_file_names.items():
    path = DATA_DIR / name
    df = pd.read_csv(path)
    print(split, path, df.shape)
    print(df["label"].value_counts().sort_index().to_dict())

In [ ]:
# Create Colab configs that allow Hugging Face model download.
# The committed configs keep downloads off by default for tests and offline reproducibility.

COLAB_CONFIG_DIR = REPO_DIR / "outputs" / "colab_configs"
COLAB_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

def make_colab_config(source_config, output_dir):
    src = REPO_DIR / source_config
    text = src.read_text(encoding="utf-8")
    if "allow_download" not in text:
        text = text.replace("[model.params]\n", "[model.params]\nallow_download = true\n", 1)
    lines = []
    for line in text.splitlines():
        if line.startswith('output_dir = '):
            lines.append(f'output_dir = "{output_dir.as_posix()}"')
        else:
            lines.append(line)
    text = "\n".join(lines) + "\n"
    target = COLAB_CONFIG_DIR / Path(source_config).name
    target.write_text(text, encoding="utf-8")
    return target

FROZEN_CONFIG = make_colab_config(
    "config-examples/benchmarks/dnabert2_frozen.toml",
    Path("outputs/benchmarks/dnabert2_frozen_colab"),
)
FINETUNE_CONFIG = make_colab_config(
    "config-examples/benchmarks/dnabert2_finetune.toml",
    Path("outputs/benchmarks/dnabert2_finetune_colab"),
)

print("Frozen config:", FROZEN_CONFIG)
print("Fine-tune config:", FINETUNE_CONFIG)

In [ ]:
# Run frozen DNABERT2 embedding benchmark.
# This is the first DNABERT2 result to compare against CNN-v2.

from seqtrainer.benchmarks.runner import run_benchmark

frozen_result = run_benchmark(FROZEN_CONFIG, base_dir=REPO_DIR, allow_skip=False)
print("status:", frozen_result.status)
print("output_dir:", frozen_result.output_dir)

In [ ]:
# Inspect frozen DNABERT2 metrics.

frozen_metrics = pd.read_csv(frozen_result.output_dir / "metrics.csv")
display(frozen_metrics)

history_path = frozen_result.output_dir / "history.csv"
if history_path.exists():
    history = pd.read_csv(history_path)
    display(history.tail())

print("Use validation MCC for model selection, then read the test row for final reporting.")

In [ ]:
# Optional: run full DNABERT2 fine-tuning.
# Keep this False until the frozen run completes and you have enough GPU time.

RUN_FINE_TUNE = False

if RUN_FINE_TUNE:
    finetune_result = run_benchmark(FINETUNE_CONFIG, base_dir=REPO_DIR, allow_skip=False)
    print("status:", finetune_result.status)
    print("output_dir:", finetune_result.output_dir)
    display(pd.read_csv(finetune_result.output_dir / "metrics.csv"))
    hist = pd.read_csv(finetune_result.output_dir / "history.csv")
    display(hist.tail())
else:
    print("Fine-tuning skipped. Set RUN_FINE_TUNE = True after frozen DNABERT2 is working.")

In [ ]:
# Optional comparison helper.
# Add CNN output folders here if you copied or generated them in the same runtime.

from seqtrainer.benchmarks.compare import compare_benchmark_runs

candidate_dirs = [frozen_result.output_dir]
if "finetune_result" in globals():
    candidate_dirs.append(finetune_result.output_dir)

comparison_dir = REPO_DIR / "outputs" / "benchmarks" / "dnabert2_colab_comparison"
comparison = compare_benchmark_runs(candidate_dirs, output_dir=comparison_dir)
print("comparison_dir:", comparison.output_dir)
display(pd.read_csv(comparison.output_dir / "comparison_metrics.csv"))